In [1]:
from embedder import Embedder

embed = Embedder()


q0 = "How does approximate nearest neighbor search work??"
# # q2 = "How to install Docker on Windows?"
# # d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v = embed.encode(q0)
# v2 = embed.encode(q2)
# dv = embed.encode(d)
print(v.shape)
print(v[0])


2026-06-26 14:36:00.884793309 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


(384,)
-0.01016662288857993


In [2]:
import sys
print(sys.executable)

import gitsource
print("gitsource ok")

/workspaces/llm-zoomcamp-2026-code/llm-zoomcamp-onnx/llm-zoomcamp-hw2/.venv/bin/python
gitsource ok


In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
len(documents)

72

In [4]:
print(v.shape)
print(v[0])

(384,)
-0.01016662288857993


In [5]:
from embedder import Embedder

embed = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embed.encode(query)

filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

doc = next(d for d in documents if d["filename"] == filename)

doc_vector = embed.encode(doc["content"])

score = doc_vector.dot(v)

print(score)

0.36107027225589694


In [6]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [7]:
from gitsource import chunk_documents
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

chunk_texts = [chunk["content"] for chunk in chunks]

chunk_vectors = embed.encode_batch(chunk_texts)

X = np.array(chunk_vectors)

scores = X.dot(v)

best_idx = scores.argmax()
best_chunk = chunks[best_idx]

print("Best score:", scores[best_idx])
print("Filename:", best_chunk["filename"])
print("Start:", best_chunk.get("start"))

Best score: 0.6489017718578813
Filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Start: 1000


In [8]:
from minsearch import VectorSearch
import numpy as np

# 1. Embed all chunks
chunk_texts = [chunk["content"] for chunk in chunks]
chunk_vectors = embed.encode_batch(chunk_texts)

X = np.array(chunk_vectors)

# 2. Build the vector search index
index = VectorSearch(keyword_fields=["filename"])
index.fit(X, chunks)

# 3. Embed the Q4 query
query = "What metric do we use to evaluate a search engine?"
q = embed.encode(query)

# 4. Search
results = index.search(q, num_results=5)

# 5. Check the first result
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [9]:
from minsearch import Index, VectorSearch
import numpy as np

query = "How do I store vectors in PostgreSQL?"

# Vector search
chunk_texts = [chunk["content"] for chunk in chunks]
chunk_vectors = embed.encode_batch(chunk_texts)
X = np.array(chunk_vectors)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

q = embed.encode(query)
vector_results = vector_index.search(q, num_results=5)

# Text search
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

text_results = text_index.search(query, num_results=5)

In [10]:
print("VECTOR RESULTS")
for r in vector_results:
    print(r["filename"], r.get("start"))

print("\nTEXT RESULTS")
for r in text_results:
    print(r["filename"], r.get("start"))

VECTOR RESULTS
02-vector-search/lessons/08-pgvector.md 0
02-vector-search/lessons/08-pgvector.md 3000
03-orchestration/lessons/05-rag.md 2000
02-vector-search/lessons/08-pgvector.md 1000
02-vector-search/lessons/08-pgvector.md 2000

TEXT RESULTS
02-vector-search/lessons/02-embeddings.md 4000
03-orchestration/lessons/05-rag.md 1000
02-vector-search/lessons/01-intro.md 0
03-orchestration/lessons/05-rag.md 0
02-vector-search/lessons/01-intro.md 1000


In [11]:
vector_files = {r["filename"] for r in vector_results}
text_files = {r["filename"] for r in text_results}

vector_only = vector_files - text_files

print(vector_only)

{'02-vector-search/lessons/08-pgvector.md'}


In [12]:
query = "How do I give the model access to tools?"

# Vector search
q = embed.encode(query)
vector_results = vector_index.search(q, num_results=5)

# Text search
text_results = text_index.search(query, num_results=5)

In [13]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [14]:
results = rrf([vector_results, text_results])

for r in results:
    print(r["filename"], r.get("start"))

01-agentic-rag/lessons/13-function-calling.md 4000
01-agentic-rag/lessons/01-intro.md 2000
01-agentic-rag/lessons/14-agentic-loop.md 0
04-evaluation/lessons/02-ground-truth.md 1000
01-agentic-rag/lessons/16-other-frameworks.md 0
